# Deliverable 3
* See the [GSoC fork of `ioos_qc`](https://github.com/Klankers/ioos_qc/) for the code implemented following the research and development in this notebook.
* See the [IOOS GSoC project proposal](https://github.com/ioos/gsoc/issues/102) for more details on origin.

**Soft end date of October 22**
*Complete assessment of a subset of tests.*

| Stage | Stage specifications | Time required | Start-End |
| ----- | -------------------- | ------------- | --------- |
| 1 | Learn about the community and note shortcomings or additional toolboxes, such that the wheel need not be reinvented. | 3 weeks | May 1 - May 24 |
| 2 | Begin test assessments and complete Group 1 “required” tests. | 14 weeks | May 25 - Sept 3 |
| 3 | Assess immediately applicable "follow up" tests. | 7 weeks | Sept 4 - Oct 22 |
| 4 | Finalize documentation. | 2 weeks | Oct 23 - Nov 2 |

Final submission of the extended project due the week of November 2nd. Note that this should include a Jupyter notebook submission to the [IOOS code lab](https://ioos.github.io/ioos_code_lab/content/intro.html).

Following a recent meeting with my mentors, we agreed that a huge amount of learning, networking, and coding has been done on the toolbox, though it has come with a cost of increased time spent on the "required" tests. While it is encouraging to see the updates from this project and the value gained from networking directly with the toolbox's users, we need to constrain the remainder of the tests to whatever is manageable.

As such, we're constraining the number of tests based on priority:
* **Density inversion test** ("suggested" test) - high similarity to the previous "pressure" test, making for a good segue following a recent hack week where I didn't manage to work on the toolbox at all.
* **Climatology test** ("strongly recommended") - potentially the most complicated test, structured unlike the others within `qartod.py`. It has a large user base, though potentially few of them are using it as intended. Improving the codebase for it is therefore a priority to 1) guarantee it isn't a barrier for future users and 2) ensure that it is working as intended.

*Any tests below would be considered a bonus*
* **Spike test** ("strongly recommended") - recent interactions with QARTOD users have described it as ineffective.
* Checking toolbox contents: It was a surprise when the timing and syntax tests were missing. Are the remaining tests in the manual present in the codebase (`qartod.py` or `argo.py`)? Prioritize those that are not.
    * Absent from the code (prioritizing):
        * **Multi-Variate** ("suggested")
        * **Previous profile** ("suggested")
        * **TS curve/space test** ("suggested")
    * Present in the code (deprioritizing):
        * **Rate of change** ("strongly recommended")
        * **Flat line** ("strongly recommended")
        * **Attenuated signal** ("suggested")

## Imports
Begin the deliverables notebook with essential imports. In theory, this should be all you need to run the tests.
* typing
* data wrangling and visualization
* dataset imports
* the QARTOD flags

I might include this Python cell with the `code` or `actions` sections of each test below, such that I don't need to rerun the whole notebook. By the end of deliverable 2, the notebook took several minutes to run and may have gotten a little bloated.

In [ ]:
from collections.abc import Sequence
from numbers import Real
import warnings
import numpy as np
import xarray as xr
fpath = "/home/aaron-mau/Data/OG1/delayed_SEA056_M102.nc"
ds = xr.load_dataset(fpath)
class QartodFlags:
    """Primary flags for QARTOD."""
    GOOD = 1
    UNKNOWN = 2
    SUSPECT = 3
    FAIL = 4
    MISSING = 9
FLAGS = QartodFlags  # Default name for all check modules
NOTEVAL_VALUE = QartodFlags.UNKNOWN

## Density Inversion Test
* How is it described in the manual
* How is it represented in the code
* Does anything need to be done in either?

### Manual description
As mentioned before, this segues well after working on the `pressure increasing` test for `argo.py` and `pressure test` for `qartod.py` prior to OHW'26.

This is described as a **suggested** test, so by itself it isn't the highest priority. This is the first test of this class to be worked on and the heirarchy of tests would best summarize them as "optional". There isn't a genuine "description" of these tests outside of their classification, so it isn't quite clear what the user gains from each. From my summarization of the manual thus far, I'd describe them as follows:
* Required: These are, well, required. If you don't do them you could expect major problems. Big data gaps. Broken data strings or positional data. Core sensors being broken and carrying over to the rest of the sensor package.
* Strongly recommended: These are some of the essential tests that, while not required, provide solid insight. Despiking, checking for flat lines, etc. are all high value tests for just about any glider sensor.
* Suggested: These are nice to pair on the side of any of the other tests. These pull more from oceanographic context and scientific insight than strict signals or statistics.

This test is essentially pairing density to pressure and confirming that they are both increasing together. This form of density is referred to as *potential* density or $\sigma_\Theta$, which if I remember correctly, requires a derived conservative temperature and absolute salinity. Yep, looks like that's what [GSW lists in the `density` module](https://teos-10.github.io/GSW-Python/density.html) (see `gsw.density.sigma0). I'm *assuming* this is referenced to 0 dbar - but the last sentance suggests that operators should be able to choose other $\sigma_t$.

Anyway, the test then says:

> When vertical profile data are obtained, this test is used to flag failed T, C, and S observations, which yield densities that do not sufficiently increase with pressure.

Which suggests that it should be returning a series of flags for those parameters. It isn't quite clear if this flag is demarcating "this *index* is bad" or "*temperature* is bad at index". The wording suggests that it *could* require looking into which factor is contributing to a bad density reading. I should add that I think we're assuming pressure is good, since it isn't mentioned in the sentance above. It's most likely that this test is intended to flag an index of the dataset, not individual components.

There is supposed to be an operator-selected density threshold that allows for some density exceptions, which is denoted as `DT` for sequential sample increments, n-1 to n. So the first point won't be flagged by this, but the last one will. Similar to the `pressure test`, this needs to work on downglides/downcasts and upglides/upcasts, so we may need a keyword argument to distinguish the sign of the cast. It says "...results produced in real time" and I'm not sure what that means, since I don't stream data or have experience with the live feed functionality of `IOOS_QC`.

The manual then says:

> From a computational point of view, this test is similar to the rate of change test (test 8). The same code can be used for both, using different variables and thresholds. As with the rate of change test, it is not known which side of the step is good versus bad.

By saying "The same code can be used for both", I'm suddenly questioning my work on the entire toolbox. Should we have been reusing the "gross_range_test" on a d/dt of the `TIME` coordinate? I think this is just another example of the manual not being correctly descriptive, as the `rate of change test` is written to describe a time series, which is not the same as $\delta \sigma_\Theta \over \delta p$. The sentance thereafter is also perhaps in error - as the previous paragraph clearly defines an index $n$ which is being examined. I would assume that, based on the previous flag ($flag_{n-1}$) you should know if the previous point was good or bad.

The manual references TEOS-10 for the defitinion of $\sigma_\Theta$, so we'll keep an eye out for `gsw` in the codebase.

This test, like many others, doesn't list a flag 9 (MISSING) but we'll want to include it with the PASS and FAIL flags. The test is described as a fail on a point $n$ if it smaller than the previous point plus the density threshold. It's weird that there is no codable condition for "SUSPECT" and it isn't described in the manual - I could envision it being like the `range test` where there is a SUSPECT bound and a FAIL bound.

> $\sigma_{\theta n-1} + DT > \sigma_{\theta n}$ flags as 4

Otherwise flag as 1. Note that the equal sign is considered 1 or PASS.

The manual wraps up with an example where DT = 0.03 kg/m^3. This aligns with what `gsw` would likely output for a value of $\sigma_\theta$.

### Code
As mentioned before, this code already exists in the toolbox.

Interestingly, this takes in the potential water density as `inp` and the pressure *or* depth values as `zinp`. The manual explicitly mentions pressure, not depth, which is similar but not identical. This test is therefore assuming that I'm doing these derivations myself prior to using it. I think this makes sense, but it might be worth clarifying suggested ways to get these params or standardizing what gets passed in. `inp` is *only* the potential density data, so as I assumed in the manual portion, we're just going to be looking at the index that is failing and not decomposing back into the source C, T, and S values to assign flags on who we think is wrong. It says that the operator should flag both temperature and salinity on the result of this test...? Is that in the TS manual that was referenced in the docstring?

This test *does* inlcude suspect and fail thresholds, rather than just one value of DT. These thresholds are most clearly described in the second line of each of the parameters in the docstring - it's the *variation* to be tolerated.

It says that it works on downcasts, upcasts, and down/up in real time but it doesn't need to be clarified in a keyword argument the way I needed to do the `pressure test`. I think this is all coming into effect in the `delta` line, where it combines with the orientation of `zinp`. However, we needed the sequence of values for the helper function in the `pressure test` to track the most extreme values. Here, we're just doing the diff twice. So it could probably get by without it for this application.

```python
delta = np.sign(np.diff(zinp)) * np.diff(inp)
```

It confirms that the arrays are the same sizes and builds a flag masked array the same way as we normally would, it's just missing the NaN mask. So we'll want to get that added.

Then it checks the input sizes to make sure that it can run something - if there's no data or only one point, it can't really do anything. So it returns a blank array or an UNKNOWN flag. Makes sense.

Finally, the thresholds we care so much about. It starts with the suspect threshold and then moves on to the fail threshold with essentially the same logic. Flagging in this order guarantees that FAIL will overwrite SUSPECT, which is what we want.

Then it... oh. Ok, so it does do a NaN mask at the end. Does that make a difference? I wonder if we could remove the `with np.errstate(invalid="ignore"):` line by simply moving it forward and doing the `valid` index checks the way we did for tests in deliverable 2.

In [ ]:
def existing_density_inversion_test(
    inp: Sequence[Real],
    zinp: Sequence[Real],
    suspect_threshold: float | None = None,
    fail_threshold: float | None = None,
) -> np.ma.core.MaskedArray:
    """With few exceptions, potential water density will increase with increasing pressure. When
    vertical profile data is obtained, this test is used to flag as failed T, C, and SP observations, which
    yield densities that do not sufficiently increase with pressure. A small operator-selected density
    threshold (DT) allows for micro-turbulent exceptions. This test can be run on downcasts, upcasts,
    or down/up cast results produced in real time.

    Both Temperature and Salinity should be flagged based on the result of this test.

    Ref: Manual for Real-Time Quality Control of in-situ Temperature and Salinity Data, Version 2.0, January 2016

    Parameters
    ----------
    inp
        Potential density values as a numeric numpy array or a list of numbers.
    zinp
        Corresponding depth/pressure values for each density.
    suspect_threshold
        A float value representing a maximum potential density(or sigma0)
        variation to be tolerated, downward density variation exceeding this will be flagged as SUSPECT.
    fail_threshold
        A float value representing a maximum potential density(or sigma0)
        variation to be tolerated, downward density variation exceeding this will be flagged as FAIL.

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input.

    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        inp = np.ma.masked_invalid(np.array(inp).astype(np.float64))
        zinp = np.ma.masked_invalid(np.array(zinp).astype(np.float64))

    # Make sure both inputs are the same size.
    if inp.shape != zinp.shape:
        msg = f"Density ({inp.shape}) and depth ({zinp.shape}) must be the same shape"
        raise ValueError(msg)

    # Start with everything as passing
    flag_arr = QartodFlags.GOOD * np.ma.ones(inp.size, dtype="uint8")

    # If no data or just one record, return respectively an empty mask array or UNKNOWN
    if inp.size == 0:
        return np.ma.masked_array([])
    inp_size = 2
    if inp.size < inp_size:
        flag_arr[0] = QartodFlags.UNKNOWN
        return flag_arr

    # Compute the vertical density variability along zinp and flip delta according to zinp variation direction
    delta = np.sign(np.diff(zinp)) * np.diff(inp)

    if suspect_threshold is not None:
        with np.errstate(invalid="ignore"):
            is_suspect = delta < suspect_threshold
            if any(is_suspect):
                flag_arr[:-1][is_suspect == True] = QartodFlags.SUSPECT  # noqa:E712 - Previous value
                flag_arr[1:][is_suspect == True] = QartodFlags.SUSPECT  # noqa:E712 - Reversed value

    if fail_threshold is not None:
        with np.errstate(invalid="ignore"):
            is_fail = delta < fail_threshold
            if any(is_fail):
                flag_arr[:-1][is_fail == True] = QartodFlags.FAIL  # noqa:E712 - Previous value
                flag_arr[1:][is_fail == True] = QartodFlags.FAIL  # noqa:E712 - Reversed Value

    # If the value or depth is masked set the flag to MISSING for this record and the following one.
    is_missing = inp.mask | zinp.mask
    flag_arr[is_missing] = QartodFlags.MISSING
    flag_arr[1:][is_missing[:-1]] = QartodFlags.MISSING
    return flag_arr

### Actions
1. Check out the BP that was mentioned in the docstring. I think [this is the most recent one](https://cdn.ioos.noaa.gov/media/2020/03/QARTOD_TS_Manual_Update2_200324_final.pdf)? This differs from the glider manual - there are a different number of tests and the test description here is slightly different.
2. Test it out on some real data. Make sure that it works on the downcast, upcast, and whatever the down/up together means.
3. Try creating the `valid` indexes and creating a NaN mask the way we did before. 

#### Reviewing this other BP
Ok, so regarding the other BP, it isn't actually too different. At least where the test is brought up.